# 08. RAG 파이프라인의 숨은 판단 — 게이트 3개를 decision model로

RAG가 하는 일을 쪼개 보면 성격이 다른 **두 종류의 일**이 섞여 있습니다.

- **순위 매기기(ranking)** — "후보 20개 중 어느 문서가 더 관련 있나". 후보 사이의 섬세한 상대 비교로, 전용 cross-encoder reranker의 영역입니다.
- **판단(judgment)** — 입력 하나에 예/아니오로 답하는 일. 표준 RAG 파이프라인에는 이런 판단이 최소 3곳에 박혀 있습니다.

| 게이트 | 질문 | 대응하는 공개 연구 패턴 |
|---|---|---|
| G1 검색 필요성 | "이 질문에 문서 검색이 필요한가?" | Adaptive-RAG |
| G2 문서 필터 | "이 chunk를 LLM에 넘길 가치가 있나?" | CRAG |
| G3 근거 검증 | "생성된 답변이 문서에 근거하는가?" | Self-RAG, Bedrock Guardrails contextual grounding check |

이 판단들을 LLM 호출로 처리하면 판단 하나에 0.5~3초가 들어 누적 오버헤드가 생성 본체보다 커지고, 비싸다고 생략하면 불필요한 검색·쓰레기 chunk·근거 없는 답변이 그대로 통과합니다. 이 노트북에서는 **세 게이트를 각각 Haiku(LLM 판정)와 strands-decider decision model로 실행**해 품질과 latency를 비교하고, 마지막에 게이트 3개를 전부 켠 파이프라인을 end-to-end로 실측합니다.

**응답 생성은 양쪽 모두 Sonnet 고정**입니다 — 01~04 노트북과 같은 설계로, 바꾸는 것은 판단 backend뿐입니다.

> 역할 분담에 주의하세요: 같은 실측(KorQuAD 300 쿼리)에서 **rerank 슬롯 자체는 decision model이 전용 cross-encoder를 이기지 못했습니다**(R@1 0.863 vs 0.887). decision model의 자리는 ranking이 아니라 판단 3곳입니다 — 자세한 수치는 모듈 콘텐츠 페이지를 참고하세요.


In [ ]:
import sys, time, urllib.request; from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # ws_common 경로 (repo 내 상대 위치)
import ws_common as W
print("decision server:", f"http://{W.STRANDS_HOST}:{W.STRANDS_PORT}")
print("health:", urllib.request.urlopen(f"http://{W.STRANDS_HOST}:{W.STRANDS_PORT}/health", timeout=5).status)
client, sonnet = W.make_clients()
haiku   = W.HaikuDecision(client, W.BEDROCK_MODEL_IDS["haiku-4-5"], W.PRICING["haiku-4-5"])
strands = W.SystemOneDecision()
DECIDERS = [("Haiku", haiku), ("StrandsDecider", strands)]

## 1. 미니 지식 베이스와 검색기

가상의 "KB한빛" 은행 FAQ 12건이 지식 베이스입니다(수치·절차 전부 합성). 검색기는 문자 bigram 겹침 기반의 가장 단순한 lexical 검색기로 고정했습니다 — 실무라면 벡터 검색 + reranker를 쓰겠지만, **이 실습의 주제는 검색이 아니라 검색 전후의 "판단"** 이기 때문입니다.

In [ ]:
from IPython.display import Markdown, display
rows = ["| id | 제목 | 본문(앞 45자) |", "|---|---|---|"]
for p in W.rag_passages():
    rows.append(f"| {p['id']} | {p['title']} | {p['text'][:45]}... |")
display(Markdown("\n".join(rows)))

q = W.rag_queries()[3]["q"]   # 해외 ATM 출금 한도
print(f"\n검색 데모: {q}")
for p, score in W.rag_retrieve(q, k=3):
    print(f"  {score:5.2f}  [{p['id']:<14s}] {p['title']}")

## 2. 게이트 1 — 검색 필요성 판정 (Adaptive-RAG)

"이 발화에 답하려면 문서 검색이 필요한가?" 모든 query가 이 게이트를 지나므로, 여기서의 지연은 **전 트래픽에 그대로 더해집니다**. 이 게이트가 없으면 "안녕하세요"에도 검색을 돌게 됩니다.

선택지 순서가 안전장치입니다 — 판정이 유효하지 않으면 `choices[0]`(=retrieve)로 폴백하므로, 오류는 항상 **검색하는 쪽(품질 안전)** 으로 갑니다.

In [ ]:
print("rubric:", W.RAG_GATE_QUESTION, "\n")
gate_stats = {}
for name, decider in DECIDERS:
    correct, lat = 0, []
    print(f"--- {name} ---")
    for item in W.rag_queries():
        d = decider.decide(state_text=item["q"], question=W.RAG_GATE_QUESTION,
                           choices=W.RAG_GATE_CHOICES, decision_id="rag_gate")
        ok = d.label == item["gate_gold"]
        correct += ok
        lat.append(d.latency_ms)
        mark = " " if ok else "X"
        print(f" {mark} [{d.label:<8s}] {d.latency_ms:>6,.0f}ms | {item['q'][:42]}")
    gate_stats[name] = (correct, sum(lat) / len(lat))
    print(f"    정확도 {correct}/{len(W.rag_queries())}, 평균 {gate_stats[name][1]:,.0f}ms\n")

## 3. 게이트 2 — 문서 필터 (CRAG)

검색 상위 chunk라고 전부 쓸모 있는 것은 아닙니다. 이 게이트는 chunk마다 "질문의 근거로 쓸모 있나?"를 yes/no로 판정합니다 — **chunk 수만큼 호출되므로 LLM 판정이면 비용이 가장 빠르게 누적되는 지점**입니다.

decision model의 고유 장점은 **확률**입니다. yes/no 텍스트가 아니라 p(yes)가 숫자로 나오므로, threshold 하나로 필터의 엄격함을 조절할 수 있습니다. reranker의 cross-encoder 점수는 비유계 raw 점수라(정답 문서 점수가 -6 ~ +10 산포) 고정 threshold로 자를 수 없습니다 — 이것이 reranker가 구조적으로 제공하지 못하는 기능입니다.

In [ ]:
item = W.rag_queries()[0]           # 해외송금 수수료 부담 방식
top = W.rag_retrieve(item["q"], k=8)
print(f"질문: {item['q']}  (정답 문서: {item['gold_id']})\n")
rows = ["| chunk | Haiku 판정 | ms | StrandsDecider p(yes) | ms |", "|---|---|---:|---:|---:|"]
kept_chars, all_chars = 0, 0
for p, score in top:
    doc = f"[{p['title']}] {p['text']}"
    dh = haiku.decide(state_text=doc, question=W.RAG_FILTER_QUESTION.format(q=item["q"]),
                      choices=W.RAG_FILTER_CHOICES, decision_id="rag_filter")
    ds = strands.decide(state_text=doc, question=W.RAG_FILTER_QUESTION.format(q=item["q"]),
                       choices=W.RAG_FILTER_CHOICES, decision_id="rag_filter")
    py = W.p_yes(ds)
    all_chars += len(doc)
    if py >= 0.5:
        kept_chars += len(doc)
    gold = " ◀ 정답" if p["id"] == item["gold_id"] else ""
    rows.append(f"| {p['id']}{gold} | {dh.label} | {dh.latency_ms:,.0f} "
                f"| {py:.3f} | {ds.latency_ms:,.0f} |")
display(Markdown("\n".join(rows)))
print(f"\nStrandsDecider p(yes)>=0.5 필터: context {all_chars:,} -> {kept_chars:,}자 "
      f"({100 * (1 - kept_chars / all_chars):,.0f}% 절감)")

## 4. 게이트 3 — 근거 검증 (Self-RAG / contextual grounding check)

생성된 답변을 내보내기 전 마지막 관문: "답변의 사실 주장이 전부 문서에서 확인되는가?" 아래 4건 중 2건은 근거 있는 답변, 2건은 **문서와 모순되는 수치·규정을 섞은 답변**입니다. 폴백 방향은 no(차단) — 게이트 오류가 hallucination을 통과시키지 않도록.

In [ ]:
docs = {p["id"]: p for p in W.rag_passages()}
rows = ["| 근거 문서 | 정답 | Haiku | ms | StrandsDecider (conf) | ms |", "|---|---|---|---:|---|---:|"]
for probe in W.rag_ground_probes():
    state = (f"문서 발췌:\n{docs[probe['doc_id']]['text']}\n\n답변:\n{probe['answer']}")
    dh = haiku.decide(state_text=state, question=W.RAG_GROUND_QUESTION,
                      choices=W.RAG_GROUND_CHOICES, decision_id="rag_ground")
    ds = strands.decide(state_text=state, question=W.RAG_GROUND_QUESTION,
                       choices=W.RAG_GROUND_CHOICES, decision_id="rag_ground")
    rows.append(f"| {probe['doc_id']} | {probe['gold']} | {dh.label} | {dh.latency_ms:,.0f} "
                f"| {ds.label} ({ds.confidence:.2f}) | {ds.latency_ms:,.0f} |")
display(Markdown("\n".join(rows)))

## 5. End-to-end — 게이트 3개를 전부 켠 파이프라인

이제 10개 질의 전부를 `G1 게이트 → (검색 → G2 필터 → Sonnet 생성) → G3 근거 검증` 파이프라인으로 흘립니다. 두 arm의 차이는 **판단 backend뿐**입니다(검색기·생성 모델·프롬프트 동일).

In [ ]:
def run_rag(decider, arm_name, top_k=4):
    docs = {p["id"]: p for p in W.rag_passages()}
    n_j, judge_ms, gen_ms, gate_ok, ground_n = 0, 0.0, 0.0, 0, 0
    t0 = time.perf_counter()
    for item in W.rag_queries():
        # G1: 검색 필요성
        d = decider.decide(state_text=item["q"], question=W.RAG_GATE_QUESTION,
                           choices=W.RAG_GATE_CHOICES, decision_id="rag_gate")
        n_j += 1; judge_ms += d.latency_ms
        gate_ok += (d.label == item["gate_gold"])
        kept = []
        if d.label == "retrieve":
            # G2: chunk별 문서 필터
            for p, _ in W.rag_retrieve(item["q"], k=top_k):
                f = decider.decide(state_text=f"[{p['title']}] {p['text']}",
                                   question=W.RAG_FILTER_QUESTION.format(q=item["q"]),
                                   choices=W.RAG_FILTER_CHOICES, decision_id="rag_filter")
                n_j += 1; judge_ms += f.latency_ms
                if f.label == "yes":
                    kept.append(p)
        ctx = "\n\n".join(f"[{p['title']}] {p['text']}" for p in kept)
        user = f"문서 발췌:\n{ctx}\n\n고객 질문: {item['q']}" if kept else item["q"]
        r = sonnet.generate(system=W.RAG_GEN_SYSTEM, user=user, max_tokens=300)
        gen_ms += r.latency_ms
        verdict = "-"
        if kept:
            # G3: 근거 검증
            g = decider.decide(state_text=f"문서 발췌:\n{ctx}\n\n답변:\n{r.text}",
                               question=W.RAG_GROUND_QUESTION,
                               choices=W.RAG_GROUND_CHOICES, decision_id="rag_ground")
            n_j += 1; judge_ms += g.latency_ms
            verdict = g.label
            ground_n += (g.label == "yes")
        gold_kept = "o" if (item["gold_id"] in {p["id"] for p in kept}) else ("-" if not item["gold_id"] else "X")
        print(f"  [{d.label:<8s}] chunk {len(kept)}개(정답 {gold_kept}) ground={verdict:<3s} | {item['q'][:38]}")
    e2e = (time.perf_counter() - t0) * 1000.0
    print(f"  >>> [{arm_name}] 판단 {n_j}회 {judge_ms:,.0f}ms | 생성 {gen_ms:,.0f}ms | e2e {e2e:,.0f}ms")
    return {"arm": arm_name, "n_queries": len(W.rag_queries()), "n_judgments": n_j,
            "judge_ms": judge_ms, "gen_ms": gen_ms, "e2e_ms": e2e,
            "gate_correct": gate_ok, "ground_pass": ground_n}

In [ ]:
print("[Haiku arm]")
res_haiku = run_rag(haiku, "Haiku")

In [ ]:
print("[StrandsDecider arm]")
res_strands = run_rag(strands, "StrandsDecider")

In [ ]:
summary = W.rag_report(res_haiku, res_strands)

### 관찰 포인트

- **e2e 차이 = 판단 몫**: 생성은 양 arm 모두 Sonnet이므로, e2e에서 벌어진 시간이 곧 판단 오버헤드입니다. LLM 판정은 chunk 필터처럼 호출이 chunk 수에 비례하는 지점에서 가장 빠르게 누적됩니다.
- **게이트 품질은 대등**: 이 소표본에서 두 backend의 게이트 판정은 대체로 일치합니다. 대규모 실측(KorQuAD 300 쿼리)에서도 검색 필요성 0.950 vs 0.940, groundedness 0.957 vs 0.957로 대등했습니다 — 품질을 양보하고 속도를 사는 거래가 아닙니다.
- **확률이라는 운영 손잡이**: 3절의 p(yes) threshold를 올리면 필터가 엄격해지고(토큰 절감↑, 놓칠 위험↑), 내리면 느슨해집니다. LLM의 yes/no 텍스트에는 이 손잡이가 없습니다.
- **역할 분담**: rerank(순위 매기기) 슬롯은 전용 cross-encoder가 우세합니다. decision model의 자리는 판단 3곳이고, ranking은 reranker, 생성은 frontier 모델이 맡는 hybrid 구성이 권장 아키텍처입니다.
- **threshold는 모델 간 이식 불가**: p(yes) 분포는 모델마다 다릅니다. decision model을 교체하면 labeled holdout으로 threshold를 재보정해야 합니다.

### 직접 해보기

- 3절의 필터 threshold를 0.5에서 0.9로 올려 보세요 — 유지되는 chunk 수와 정답 보존이 어떻게 달라지나요?
- `run_rag(..., top_k=8)`로 검색 폭을 늘려 보세요. Haiku arm의 판단 오버헤드는 top_k에 비례해 커지지만 StrandsDecider arm은 거의 그대로입니다.
- `W.rag_ground_probes()` 형식으로 자신만의 조작된 답변을 만들어 G3가 잡아내는지 확인해 보세요.
- G1 rubric(`W.RAG_GATE_QUESTION`)의 direct 기준 문구를 바꿔 보세요 — LLM 판정과 decision model 중 어느 쪽이 문구 변화에 더 민감한가요?
